# Protein Remote Homology Detection with SVM

This notebook applies **linear and RBF support vector machines** to protein remote homology detection using the SCOP 1.53 benchmark published with Liao and Noble's SVM-pairwise work.

The workflow uses only protein sequences:

1. Download the official FASTA sequences and family-wise benchmark splits.
2. Represent each sequence with normalized amino-acid 2-mer and 3-mer frequencies.
3. Train linear and RBF SVMs with class balancing.
4. Evaluate ranking performance using ROC-AUC, ROC50, and precision–recall AUC.

**Benchmark source:** [Noble Lab — SVM-pairwise supplementary data](https://noble.gs.washington.edu/proj/svm-pairwise/).

## 1. Task definition

SCOP organizes protein domains into a hierarchy that includes **class → fold → superfamily → family**. In this remote-homology benchmark, a target family is held out for testing while other members of its superfamily provide positive training examples. This makes the task harder than ordinary random sequence splitting.

The membership table uses four codes:

| Code | Meaning | Label used here |
|---|---|---|
| 1 | Positive training sequence | 1 |
| 2 | Negative training sequence | 0 |
| 3 | Positive test sequence | 1 |
| 4 | Negative test sequence | 0 |
| 0 | Not used for that family's experiment | Excluded |

The dataset contains 4,352 sequences and 54 target-family experiments. Accuracy is not emphasized because the negative class heavily outnumbers the positive class.

## 2. Imports and reproducibility

In [ ]:
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc
)

RANDOM_STATE = 42

## 3. Download the official benchmark

In [ ]:
BASE_URL = "https://noble.gs.washington.edu/proj/svm-pairwise"
DATA_DIR = Path("data/scop_1.53")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_if_missing(filename):
    destination = DATA_DIR / filename
    if not destination.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Downloading {url}")
        response = requests.get(url, timeout=(30, 180))
        response.raise_for_status()
        destination.write_bytes(response.content)
    return destination


fasta_path = download_if_missing("scop.1.53.1e-25.fasta")
membership_path = download_if_missing("all_seq_fam_membership.txt")

## 4. Parse sequences and benchmark membership

In [ ]:
def read_fasta(path):
    records = []
    sequence_id = None
    sequence_parts = []

    with open(path, encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith(">"):
                if sequence_id is not None:
                    records.append((sequence_id, "".join(sequence_parts).upper()))
                sequence_id = line[1:].split()[0]
                sequence_parts = []
            elif line:
                sequence_parts.append(line)

    if sequence_id is not None:
        records.append((sequence_id, "".join(sequence_parts).upper()))

    return pd.DataFrame(records, columns=["ID", "sequence"])


sequences = read_fasta(fasta_path)
membership = pd.read_csv(membership_path, sep=r"\s+")

assert sequences.shape == (4352, 2), "Unexpected FASTA dataset size"
assert membership.shape == (4352, 55), "Unexpected membership table size"
assert sequences["ID"].is_unique and membership["ID"].is_unique

# Keep one row per sequence and retain all 54 experiment columns.
data = sequences.merge(membership, on="ID", validate="one_to_one")
assert len(data) == 4352

display(sequences.head())
print(f"Sequences: {len(sequences):,}")
print(f"Target-family experiments: {len(membership.columns) - 1}")

## 5. Sequence representation and evaluation metrics

`TfidfVectorizer` is used here **without inverse-document-frequency weighting**. With `use_idf=False` and L2 normalization, it produces normalized character n-gram frequencies—an accessible approximation to a spectrum-style sequence representation.

ROC50 measures the ROC area up to 50 false positives, normalized to the interval `[0, 1]`. It emphasizes early retrieval, which matters when a user examines only the highest-ranked candidates.

In [ ]:
def roc50_score(y_true, scores, max_false_positives=50):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n_positive = int(y_true.sum())
    n_negative = len(y_true) - n_positive

    if n_positive == 0 or n_negative == 0:
        return np.nan

    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = y_true[order]
    true_positives = np.concatenate([[0], np.cumsum(sorted_labels)])
    false_positives = np.concatenate([[0], np.cumsum(1 - sorted_labels)])
    cutoff = min(max_false_positives, n_negative)

    keep = false_positives <= cutoff
    fp_curve = false_positives[keep]
    tpr_curve = true_positives[keep] / n_positive

    # Interpolate the true-positive rate exactly at the cutoff.
    if fp_curve[-1] < cutoff:
        next_index = len(fp_curve)
        tpr_at_cutoff = np.interp(
            cutoff,
            [fp_curve[-1], false_positives[next_index]],
            [tpr_curve[-1], true_positives[next_index] / n_positive]
        )
        fp_curve = np.append(fp_curve, cutoff)
        tpr_curve = np.append(tpr_curve, tpr_at_cutoff)

    return auc(fp_curve / cutoff, tpr_curve)

## 6. Train and evaluate both SVMs for one target family

In [ ]:
def evaluate_family(target_family):
    family_data = data[["ID", "sequence", target_family]]

    train = family_data[family_data[target_family].isin([1, 2])]
    test = family_data[family_data[target_family].isin([3, 4])]
    y_train = (train[target_family] == 1).astype(int).to_numpy()
    y_test = (test[target_family] == 3).astype(int).to_numpy()

    # Fit the feature extractor on training sequences only.
    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 3),
        lowercase=False,
        use_idf=False,
        norm="l2"
    )
    X_train = vectorizer.fit_transform(train["sequence"])
    X_test = vectorizer.transform(test["sequence"])

    models = {
        "Linear SVM": LinearSVC(
            C=1.0,
            class_weight="balanced",
            dual=True,
            max_iter=10000,
            random_state=RANDOM_STATE
        ),
        "RBF SVM": SVC(
            C=2.0,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            cache_size=512
        )
    }

    rows = []
    scores_by_model = {}
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        scores = model.decision_function(X_test)
        scores_by_model[model_name] = scores
        rows.append({
            "Model": model_name,
            "ROC-AUC": roc_auc_score(y_test, scores),
            "ROC50": roc50_score(y_test, scores),
            "PR-AUC": average_precision_score(y_test, scores),
            "Train positives": int(y_train.sum()),
            "Test positives": int(y_test.sum()),
            "Features": X_train.shape[1]
        })

    return {
        "results": pd.DataFrame(rows),
        "scores": scores_by_model,
        "y_test": y_test,
        "test": test,
        "y_train": y_train
    }


TARGET_FAMILY_INDEX = 0  # Try values from 0 through 53.
TARGET_FAMILY = membership.columns[TARGET_FAMILY_INDEX + 1]
evaluation = evaluate_family(TARGET_FAMILY)

print(f"Target SCOP family: {TARGET_FAMILY}")
display(evaluation["results"].round(4))

## 7. Plot ROC and precision–recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_test = evaluation["y_test"]

for model_name, scores in evaluation["scores"].items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    model_roc_auc = roc_auc_score(y_test, scores)
    axes[0].plot(fpr, tpr, label=f"{model_name} (AUC = {model_roc_auc:.3f})")

    precision, recall, _ = precision_recall_curve(y_test, scores)
    model_pr_auc = average_precision_score(y_test, scores)
    axes[1].plot(recall, precision, label=f"{model_name} (AP = {model_pr_auc:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC Curves")
axes[0].legend()

positive_rate = y_test.mean()
axes[1].axhline(positive_rate, color="k", linestyle="--", alpha=0.5,
                label=f"Random baseline = {positive_rate:.4f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–Recall Curves")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Inspect the highest-ranked test sequences

In [ ]:
MODEL_TO_INSPECT = "RBF SVM"

ranked_candidates = evaluation["test"][["ID", "sequence"]].copy()
ranked_candidates["remote_homolog"] = evaluation["y_test"]
ranked_candidates["decision_score"] = evaluation["scores"][MODEL_TO_INSPECT]
ranked_candidates = ranked_candidates.sort_values("decision_score", ascending=False)

display(ranked_candidates.head(15))

## 9. Optional: run all 54 benchmark families

The following cell repeats the complete training and evaluation process for every target family. It may take several minutes. The final table averages the metrics across families, which is more informative than reporting only one favorable split.

In [ ]:
# all_results = []
# for family in membership.columns[1:]:
#     family_evaluation = evaluate_family(family)
#     summary = family_evaluation["results"].copy()
#     summary.insert(0, "Family", family)
#     all_results.append(summary)
#     print(f"Completed {family}")
#
# all_results = pd.concat(all_results, ignore_index=True)
# mean_results = all_results.groupby("Model")[["ROC-AUC", "ROC50", "PR-AUC"]].mean()
# display(mean_results.round(4))
# all_results.to_csv("scop_svm_all_family_results.csv", index=False)

## 10. Interpretation and limitations

- **Use ranking metrics for this benchmark.** SVM decision scores rank likely remote homologs. A default zero threshold is not necessarily calibrated, especially under severe class imbalance.
- **Avoid leakage.** The vectorizer and SVM are fitted using training sequences only. Any hyperparameter tuning should also use cross-validation within the training set.
- **This is a sequence-only baseline.** It does not use multiple sequence alignments, profile HMMs, tertiary structures, or learned protein language-model embeddings.
- **One family is not enough.** Use the all-family loop and report mean and variability across families before drawing conclusions.
- **Fold detection is a related but different split.** To simulate fold recognition, hold out a superfamily for testing and train on other superfamilies within the same fold; do not reuse the family-holdout labels unchanged.

## References

1. Liao, L. and Noble, W. S. *Combining pairwise sequence similarity and support vector machines for detecting remote protein evolutionary and structural relationships.* [Paper and benchmark links](https://noble.gs.washington.edu/proj/svm-pairwise/).
2. Murzin, A. G. et al. *SCOP: A Structural Classification of Proteins database.* [SCOP database](https://scop.mrc-lmb.cam.ac.uk/).
3. Leslie, C. et al. *The spectrum kernel: A string kernel for SVM protein classification.* [Paper](https://academic.oup.com/bioinformatics/article/18/suppl_1/S115/255074).